 # Bitcoin testing

In [1]:
import torch

model = torch.load("BTC_final_model.pth", weights_only=False)
model.eval()

BiLSTM(
  (lstm): LSTM(29, 64, num_layers=2, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [2]:
import pandas as pd

bit = pd.read_csv("BTC-USD_last_3_year.csv")

bit = bit.iloc[2:].reset_index(drop=True)
bit.columns.values[0] = "Date"

In [3]:
bit["Date"] = pd.to_datetime(bit["Date"])

cols = ["Close", "High", "Low", "Open", "Volume"]
bit[cols] = bit[cols].apply(pd.to_numeric)

bit = bit.dropna().reset_index(drop=True)

In [4]:
from feature_engineering import calculate_features

bit = calculate_features(bit)

In [5]:
bit.columns

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'Daily_Return', 'Close_Lag_1',
       'Close_Lag_7', 'Close_Lag_14', 'Close_Lag_30', 'Close_Lag_90',
       'Close_Lag_180', 'MA_7', 'MA_21', 'MA_60', 'MA_180', 'Volatility_7',
       'Volatility_21', 'Volatility_60', 'Volatility_180', 'HL_Spread',
       'Volume_MA_7', 'Volume_MA_21', 'Volume_MA_60', 'Volume_MA_180',
       'Volume_Change', 'Target_Return_7', 'Target_Return_30',
       'Target_Return_365', 'Target_Up'],
      dtype='object')

In [6]:
X_bit = bit.drop(columns=[
    "Close"
]).values

In [7]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_scaled_bit = scaler.fit_transform(X_bit)

In [8]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_scaled_bit = scaler.fit_transform(X_bit)

In [9]:
import numpy as np

def create_sequences(X, time_steps=30):
    X_seq_bit = []
    for i in range(len(X_bit) - time_steps):
        X_seq_bit.append(X_bit[i:i+time_steps])
    return np.array(X_seq_bit)

X_seq_bit = create_sequences(X_scaled_bit)

In [10]:
preds_bit = model(torch.tensor(X_seq_bit, dtype=torch.float32)).detach().numpy()
preds_bit[:10]

array([[-0.09951445],
       [-0.02239515],
       [-0.04955575],
       [-0.08221273],
       [-0.01181264],
       [-0.08735996],
       [-0.11478302],
       [-0.11581548],
       [ 0.0244521 ],
       [-0.1153819 ]], dtype=float32)

In [13]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

y_actual = bit["Close"].values[30:]
y_pred = preds_bit.flatten()

mse = mean_squared_error(y_actual, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_actual, y_pred)
r2 = r2_score(y_actual, y_pred)
mape = np.mean(np.abs((y_actual - y_pred) / y_actual)) * 100

print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

MSE: 4741748789.03
RMSE: 68860.36
MAE: 65839.73
R2 Score: -10.6540
MAPE: 100.00%
